# Career Assistant — RAG Vector Store Builder
Run cells top to bottom. **Cell 5** is the one you repeat in slices to stay under the 50 RPM rate limit.

## Cell 1 — Imports & Config

In [1]:
import os
import time
from typing import Any, List, Sequence

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import NodeParser
from llama_index.core.schema import BaseNode, TextNode
from llama_index.core.extractors import (
    TitleExtractor,
    QuestionsAnsweredExtractor,
    SummaryExtractor,
    KeywordExtractor,
)
from llama_index.llms.anthropic import Anthropic as LlamaAnthropic

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# ── Config ─────────────────────────────────────────────────────────────────
PDF_PATH   = '../data/RAG_sample_data.pdf'
CHROMA_DIR = '../data/chroma_db_with_metadata'
DELIMITER  = '##'

# Sleep budget per chunk.
# 4 extractors x 5 s = 20 s/chunk -> ~12 RPM, well under the 50 RPM limit.
# Lower this value if you have a higher API tier.
SECONDS_BETWEEN_CHUNKS = 20



print('Imports OK')

Imports OK


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## Cell 2 — LLM, Node Parser & Extractors

In [3]:
# Docs: https://docs.llamaindex.ai/en/stable/examples/llm/anthropic/
llm = LlamaAnthropic(
    model="claude-haiku-4-5-20251001",   # fast + capable; swap for haiku (cheaper) or opus (most powerful)
    temperature=0.1,              # low temp for deterministic metadata extraction
    max_tokens=1024,
)

# Docs: https://docs.llamaindex.ai/en/stable/module_guides/loading/documents_and_nodes/
class DelimiterNodeParser(NodeParser):
    """Splits each LlamaIndex Document on '##', one TextNode per role."""

    delimiter: str = DELIMITER

    @classmethod
    def from_defaults(cls, delimiter: str = DELIMITER, **kwargs: Any) -> 'DelimiterNodeParser':
        return cls(delimiter=delimiter, **kwargs)

    def _parse_nodes(
        self,
        nodes: Sequence[BaseNode],
        show_progress: bool = False,
        **kwargs: Any,
    ) -> List[BaseNode]:
        result: List[BaseNode] = []
        for node in nodes:
            normalised = node.get_content().replace('\n', ' ')
            for i, chunk in enumerate(normalised.split(self.delimiter)):
                chunk = chunk.strip()
                if not chunk:
                    continue
                result.append(TextNode(
                    text=chunk,
                    metadata={**node.metadata, 'chunk_index': i},
                    relationships=node.relationships,
                ))
        print(f"  [DelimiterNodeParser] {len(result)} chunks from delimiter '{self.delimiter}'")
        return result

# Docs: https://docs.llamaindex.ai/en/stable/module_guides/indexing/metadata_extraction/
extractors = [
    ('TitleExtractor',             TitleExtractor(nodes=5, llm=llm)),
    ('QuestionsAnsweredExtractor', QuestionsAnsweredExtractor(questions=3, llm=llm)),
    ('SummaryExtractor',           SummaryExtractor(summaries=['self'], llm=llm)),
    ('KeywordExtractor',           KeywordExtractor(keywords=8, llm=llm)),
]

sleep_per_call = SECONDS_BETWEEN_CHUNKS / len(extractors)
print(f'sleep_per_call = {sleep_per_call:.0f}s')
print('LLM + extractors ready')

sleep_per_call = 5s
LLM + extractors ready


## Cell 3 — Load PDF & Split into Chunks (no LLM calls)

In [4]:
documents  = SimpleDirectoryReader(input_files=[PDF_PATH]).load_data()
print(f'{len(documents)} page(s) loaded')

splitter   = DelimiterNodeParser.from_defaults(delimiter=DELIMITER)
all_chunks = splitter(documents)
print(f'{len(all_chunks)} role chunks ready\n')

# Preview all chunk boundaries so you can plan your slices
for i, c in enumerate(all_chunks):
    print(f'  [{i}] {c.text[:80].strip()!r}...')

17 page(s) loaded
  [DelimiterNodeParser] 200 chunks from delimiter '##'
200 role chunks ready

  [0] 'Data Engineer: A data engineer designs, builds, and maintains the architecture t'...
  [1] 'Data Analyst: Data analysts are tasked with interpreting and analyzing complex d'...
  [2] 'Data Scientist: Data scientists utilize advanced analytical and programming skil'...
  [3] 'Machine Learning Engineer/ ML Engineer: Machine learning engineers develop and d'...
  [4] 'Database Administrator: Database administrators are responsible for the performa'...
  [5] 'Data Architect: Data architects design and create the blueprint for managing and'...
  [6] 'Artificial Intelligence Specialist: Artificial intelligence specialists focus on'...
  [7] 'Data and Analytics Manager: Data and analytics managers oversee the strategic pl'...
  [8] 'Business Intelligence Developer/ BI developer: Business intelligence developers'...
  [9] 'DevOps Engineer  DevOps engineers are responsible for bridging the gap

## Cell 4 — Define `run_metadata_extractor()` & State
Run once. Sets up the accumulator and the helper function you will call in slices.

In [5]:
# Global accumulator — persists across all run_metadata_extractor() calls in this session
enriched_nodes: List[BaseNode] = []

def run_metadata_extractor(chunks: List[BaseNode]) -> None:
    """
    Enrich a slice of chunks with all four metadata extractors.

    Processes ONE chunk at a time and sleeps `sleep_per_call` seconds
    after every single LLM call, so the pipeline never fires more than
    ~12 requests/min regardless of how many chunks are in the slice.

    Results are appended to the global `enriched_nodes` list.

    Usage
    -----
        run_metadata_extractor(all_chunks[0:5])
        run_metadata_extractor(all_chunks[5:10])
        # ... continue until all chunks are done
    """
    global enriched_nodes

    total = len(chunks)
    print(f'Starting enrichment for {total} chunk(s)')
    print(f'Sleep between LLM calls : {sleep_per_call:.0f}s')
    print(f'Estimated time          : ~{total * SECONDS_BETWEEN_CHUNKS / 60:.1f} min\n')

    for chunk_idx, node in enumerate(chunks, 1):
        print(f'  [{chunk_idx}/{total}] {node.text[:70].strip()!r}...')
        current_nodes = [node]

        for extractor_name, extractor in extractors:
            print(f'    -> {extractor_name}', end=' ', flush=True)
            current_nodes = extractor.process_nodes(current_nodes)
            print('done')

            is_last_call = (chunk_idx == total) and (extractor_name == extractors[-1][0])
            if not is_last_call:
                time.sleep(sleep_per_call)

        enriched_nodes.extend(current_nodes)
        print(f'    chunk enriched. Total enriched so far: {len(enriched_nodes)}\n')

    print(f'Batch complete. enriched_nodes total = {len(enriched_nodes)}')

print('run_metadata_extractor() defined')
print(f'Total chunks to process : {len(all_chunks)}')
print()
print('Now run Cell 5 slices one at a time, e.g.:')
print('  run_metadata_extractor(all_chunks[0:5])')
print('  run_metadata_extractor(all_chunks[5:10])')

run_metadata_extractor() defined
Total chunks to process : 200

Now run Cell 5 slices one at a time, e.g.:
  run_metadata_extractor(all_chunks[0:5])
  run_metadata_extractor(all_chunks[5:10])


## Cell 5 — Run Enrichment in Slices

Execute **one cell at a time**. Wait for it to finish before running the next.

Adjust slice indices based on the chunk count printed in Cell 3.

> **If you still hit a 429:** go back to Cell 1, increase `SECONDS_BETWEEN_CHUNKS` to `30`, then re-run Cells 1 → 4 before continuing.

In [7]:
len(all_chunks)

200

In [6]:
run_metadata_extractor(all_chunks[0:5])

Starting enrichment for 5 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~1.7 min

  [1/5] 'Data Engineer: A data engineer designs, builds, and maintains the arch'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:28:24,844 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 16:28:26,699 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.10s/it]


done
    -> QuestionsAnsweredExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:28:31,721 - INFO - Retrying request to /v1/messages in 0.420634 seconds
2026-05-13 16:28:35,662 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.95s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:30:52,193 - INFO - Retrying request to /v1/messages in 0.485701 seconds
2026-05-13 16:30:55,922 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.74s/it]


done
    -> KeywordExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:31:00,953 - INFO - Retrying request to /v1/messages in 0.447995 seconds
2026-05-13 16:31:02,290 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:01<00:00,  1.35s/it]

done
    chunk enriched. Total enriched so far: 5

Batch complete. enriched_nodes total = 5


In [10]:
run_metadata_extractor(all_chunks[5:10])

Starting enrichment for 5 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~1.7 min

  [1/5] 'Data Architect: Data architects design and create the blueprint for ma'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:33:30,451 - INFO - Retrying request to /v1/messages in 0.444560 seconds
2026-05-13 16:33:32,850 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 16:33:35,524 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:05<00:00,  5.08s/it]


done
    -> QuestionsAnsweredExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:33:40,548 - INFO - Retrying request to /v1/messages in 0.454783 seconds
2026-05-13 16:33:44,765 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:04<00:00,  4.23s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:36:06,223 - INFO - Retrying request to /v1/messages in 0.396431 seconds
2026-05-13 16:36:08,509 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:02<00:00,  2.29s/it]


done
    -> KeywordExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:36:13,527 - INFO - Retrying request to /v1/messages in 0.485944 seconds
2026-05-13 16:36:15,182 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:01<00:00,  1.66s/it]

done
    chunk enriched. Total enriched so far: 10

Batch complete. enriched_nodes total = 10


In [11]:
run_metadata_extractor(all_chunks[10:15])

Starting enrichment for 5 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~1.7 min

  [1/5] 'releases, implement continuous integration and deployment, and ensure'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:36:15,258 - INFO - Retrying request to /v1/messages in 0.417585 seconds
2026-05-13 16:36:16,446 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 16:36:17,516 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:02<00:00,  2.27s/it]


done
    -> QuestionsAnsweredExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:36:22,535 - INFO - Retrying request to /v1/messages in 0.486828 seconds
2026-05-13 16:36:26,416 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.89s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:38:41,949 - INFO - Retrying request to /v1/messages in 0.428058 seconds
2026-05-13 16:38:44,675 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:02<00:00,  2.73s/it]


done
    -> KeywordExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:38:49,696 - INFO - Retrying request to /v1/messages in 0.377998 seconds
2026-05-13 16:38:50,932 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:01<00:00,  1.24s/it]

done
    chunk enriched. Total enriched so far: 15

Batch complete. enriched_nodes total = 15


In [12]:
run_metadata_extractor(all_chunks[15:20])

Starting enrichment for 5 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~1.7 min

  [1/5] 'Software Developer: Software developers are proficient in programming'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:38:50,994 - INFO - Retrying request to /v1/messages in 0.401656 seconds
2026-05-13 16:38:52,166 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 16:38:53,681 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:02<00:00,  2.70s/it]


done
    -> QuestionsAnsweredExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:38:58,705 - INFO - Retrying request to /v1/messages in 0.382847 seconds
2026-05-13 16:39:01,986 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.29s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:41:13,329 - INFO - Retrying request to /v1/messages in 0.470542 seconds
2026-05-13 16:41:16,719 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.40s/it]


done
    -> KeywordExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:41:21,741 - INFO - Retrying request to /v1/messages in 0.405715 seconds
2026-05-13 16:41:23,399 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:01<00:00,  1.67s/it]

done
    chunk enriched. Total enriched so far: 20

Batch complete. enriched_nodes total = 20


In [13]:
run_metadata_extractor(all_chunks[20:50])


Starting enrichment for 30 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~10.0 min

  [1/30] 'Web Developer: Web developers design, build, and maintain websites. Th'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:41:23,491 - INFO - Retrying request to /v1/messages in 0.474416 seconds
2026-05-13 16:41:25,015 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 16:41:27,721 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:04<00:00,  4.24s/it]


done
    -> QuestionsAnsweredExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:41:32,740 - INFO - Retrying request to /v1/messages in 0.470973 seconds
2026-05-13 16:41:36,755 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:04<00:00,  4.02s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

100%|██████████| 1/1 [00:03<00:00,  3.22s/it]


done
    -> KeywordExtractor 

100%|██████████| 1/1 [00:01<00:00,  1.60s/it]

done
    chunk enriched. Total enriched so far: 50

Batch complete. enriched_nodes total = 50


In [14]:
run_metadata_extractor(all_chunks[50:80])


Starting enrichment for 30 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~10.0 min

  [1/30] 'Information security frameworks: Information security frameworks provi'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 16:56:39,411 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 16:56:42,050 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.55s/it]


done
    -> QuestionsAnsweredExtractor 

100%|██████████| 1/1 [00:03<00:00,  3.79s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

100%|██████████| 1/1 [00:02<00:00,  2.34s/it]


done
    -> KeywordExtractor 

100%|██████████| 1/1 [00:00<00:00,  1.03it/s]

done
    chunk enriched. Total enriched so far: 80

Batch complete. enriched_nodes total = 80


In [15]:
run_metadata_extractor(all_chunks[80:100])


Starting enrichment for 20 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~6.7 min

  [1/20] 'VPN: VPN (Virtual Private Network) is a technology that creates a secu'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 17:11:51,196 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 17:11:52,775 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:02<00:00,  2.70s/it]


done
    -> QuestionsAnsweredExtractor 

100%|██████████| 1/1 [00:03<00:00,  3.22s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

100%|██████████| 1/1 [00:02<00:00,  2.44s/it]


done
    -> KeywordExtractor 

100%|██████████| 1/1 [00:00<00:00,  1.24it/s]

done
    chunk enriched. Total enriched so far: 100

Batch complete. enriched_nodes total = 100


In [16]:

run_metadata_extractor(all_chunks[100:120])

Starting enrichment for 20 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~6.7 min

  [1/20] 'Data security: Data security refers to the measures and protocols impl'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 17:22:00,442 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 17:22:02,554 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.15s/it]


done
    -> QuestionsAnsweredExtractor 

100%|██████████| 1/1 [00:03<00:00,  3.64s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

100%|██████████| 1/1 [00:02<00:00,  2.48s/it]


done
    -> KeywordExtractor 

100%|██████████| 1/1 [00:02<00:00,  2.77s/it]

done
    chunk enriched. Total enriched so far: 120

Batch complete. enriched_nodes total = 120


In [17]:

run_metadata_extractor(all_chunks[120:140])

Starting enrichment for 20 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~6.7 min

  [1/20] 'Operating systems: Operating systems are software that manage computer'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 17:32:19,459 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 17:32:21,221 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:02<00:00,  2.87s/it]


done
    -> QuestionsAnsweredExtractor 

100%|██████████| 1/1 [00:03<00:00,  3.58s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

100%|██████████| 1/1 [00:02<00:00,  2.15s/it]


done
    -> KeywordExtractor 

100%|██████████| 1/1 [00:00<00:00,  1.03it/s]

done
    chunk enriched. Total enriched so far: 140

Batch complete. enriched_nodes total = 140


In [18]:
run_metadata_extractor(all_chunks[140:160])

Starting enrichment for 20 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~6.7 min

  [1/20] 'Scalability: Scalability refers to the ability of a system, network, o'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 17:42:16,586 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 17:42:18,070 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:02<00:00,  2.44s/it]


done
    -> QuestionsAnsweredExtractor 

100%|██████████| 1/1 [00:02<00:00,  2.82s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

100%|██████████| 1/1 [00:03<00:00,  3.33s/it]


done
    -> KeywordExtractor 

100%|██████████| 1/1 [00:01<00:00,  1.09s/it]

done
    chunk enriched. Total enriched so far: 160

Batch complete. enriched_nodes total = 160


In [19]:
run_metadata_extractor(all_chunks[160:200])

Starting enrichment for 40 chunk(s)
Sleep between LLM calls : 5s
Estimated time          : ~13.3 min

  [1/40] 'Understanding of data modeling reporting: Understanding of data modeli'...
    -> TitleExtractor 

  0%|          | 0/1 [00:00<?, ?it/s]2026-05-13 17:52:11,772 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-13 17:52:14,119 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
100%|██████████| 1/1 [00:03<00:00,  3.65s/it]


done
    -> QuestionsAnsweredExtractor 

100%|██████████| 1/1 [00:03<00:00,  3.16s/it]


done
    -> SummaryExtractor 

...

done
    -> SummaryExtractor 

100%|██████████| 1/1 [00:02<00:00,  2.30s/it]


done
    -> KeywordExtractor 

100%|██████████| 1/1 [00:01<00:00,  1.03s/it]

done
    chunk enriched. Total enriched so far: 200

Batch complete. enriched_nodes total = 200


In [ ]:
# Add more cells as needed:
# run_metadata_extractor(all_chunks[20:25])
# run_metadata_extractor(all_chunks[25:30])

## Cell 6 — Inspect Sample Metadata

In [20]:
print(f'Total enriched nodes: {len(enriched_nodes)}\n')
print('=== Metadata sample (first node) ===')
for key, val in enriched_nodes[0].metadata.items():
    print(f'  {key}: {str(val)}')

Total enriched nodes: 200

=== Metadata sample (first node) ===
  page_label: 1
  file_name: RAG_sample_data.pdf
  file_path: ..\data\RAG_sample_data.pdf
  file_type: application/pdf
  file_size: 459876
  creation_date: 2026-05-06
  last_modified_date: 2026-05-06
  chunk_index: 0
  document_title: # Data Engineering: Core Responsibilities, Infrastructure, and Architecture

**Alternative options:**

1. **Data Engineering Fundamentals: Architecture, Infrastructure, and Operational Responsibilities**

2. **Data Engineering Essentials: Design, Infrastructure, and Data Management Operations**

3. **Comprehensive Guide to Data Engineering: Architecture, Infrastructure, and Core Functions**

4. **Data Engineering Framework: Infrastructure Design, Architecture, and Operational Practices**

---

**Recommendation:** The first option balances all three key elements (responsibilities, infrastructure, architecture) while maintaining clarity and conciseness. It's suitable for both foundational and i

## Cell 7 — Convert to LangChain Documents
Run once all slices in Cell 5 are complete.

In [21]:
def node_to_langchain_doc(node) -> Document:
    """
    Prepend extracted metadata into page_content so it is embedded
    alongside the raw text — the 'metadata-aware embedding' technique.
    """
    meta = node.metadata
    enriched_text = (
        f"[TITLE]: {meta.get('document_title', '')}\n"
        f"[SUMMARY]: {meta.get('section_summary', '')}\n"
        f"[QUESTIONS]: {meta.get('questions_this_excerpt_can_answer', '')}\n"
        f"[KEYWORDS]: {meta.get('excerpt_keywords', '')}\n\n"
        f"[CONTENT]: {node.get_content()}"
    )
    return Document(
        page_content=enriched_text,
        metadata={
            'source':    PDF_PATH,
            'node_id':   node.node_id,
            'title':     meta.get('document_title', ''),
            'summary':   meta.get('section_summary', ''),
            'questions': meta.get('questions_this_excerpt_can_answer', ''),
            'keywords':  meta.get('excerpt_keywords', ''),
        },
    )

langchain_docs = [node_to_langchain_doc(n) for n in enriched_nodes]
print(f'{len(langchain_docs)} LangChain Documents ready')

200 LangChain Documents ready


## Cell 8 — Store in ChromaDB

In [22]:
# Docs: https://python.langchain.com/docs/integrations/vectorstores/chroma/
embeddings  = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
)
vectorstore.add_documents(langchain_docs)
print('All enriched chunks stored in ChromaDB')

2026-05-13 18:18:35,813 - INFO - No device provided, using cpu
2026-05-13 18:18:36,191 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 18:18:36,193 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-13 18:18:36,202 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-13 18:18:36,247 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 18:18:36,256 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/con

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-05-13 18:18:36,863 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-13 18:18:36,904 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-13 18:18:36,947 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-13 18:18:36,988 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-13 18:18:37,029 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 18:18:37,036 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/senten

All enriched chunks stored in ChromaDB


## Cell 9 — Smoke Test

In [24]:
results = vectorstore.similarity_search('Data science career growth opportunities', k=2)
for i, r in enumerate(results, 1):
    print(f'\n[Result {i}]')
    print(r.page_content[:400])
    print('Metadata:', {k: v[:80] if isinstance(v, str) else v
                        for k, v in r.metadata.items()})


[Result 1]
[TITLE]: # Data Analyst: Essential Technical Skills, Tools, and Core Competencies

or more concisely:

# Data Analyst Technical Skills and Competencies Guide

---

**Rationale:** The comprehensive title combines the key elements from both options:
- Acknowledges the **role** (Data Analyst)
- Incorporates **technical skills** (programming, statistical expertise)
- References **tools** (visualizatio
Metadata: {'title': '# Data Analyst: Essential Technical Skills, Tools, and Core Competencies\n\nor mor', 'questions': '# 3 Questions This Context Can Answer\n\n1. **What are the core technical skills a', 'node_id': '5cc1eec4-beb6-4e0f-920a-9b3a79df30aa', 'keywords': '# Keywords:\n\nData Analyst, SQL, Python, R, Tableau, Power BI, Statistical Analys', 'source': '../data/RAG_sample_data.pdf', 'summary': '# Summary of Key Topics and Entities\n\n## Key Topics:\n- **Role Definition**: Data'}

[Result 2]
[TITLE]: # Data Science: Skills, Methods, and Business Applications

**Rationale:*